# 03. 분류 모델 (지도 학습)

**목적**: 5개 데이터를 통합하여 건강 수치 기반으로 영양소 기준 및 식단 플랜 예측

**입력 피처**: Age, Gender, Height, Weight, BMI, sbp, dbp, fbs, waist, group  
**타겟**: Recommended_Calories, Recommended_Protein, Recommended_Carbs, Recommended_Fats, Recommended_Meal_Plan  
**모델**: Random Forest (NaN 자체 처리 가능)

## 0. 라이브러리 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
import joblib
import os
import warnings

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, accuracy_score, f1_score, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')

## 1. Google Drive 마운트 및 데이터 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/AH_03_06'

In [ ]:
# ── 1-1. 국민건강보험공단 ──────────────────────────────────────
nhis = pd.read_csv(f'{BASE}/data/raw/국민건강보험공단_건강검진정보_2024.CSV.csv', encoding='cp949')
nhis = nhis.rename(columns={
    '성별코드': 'gender',
    '연령대코드(5세단위)': 'age',
    '신장(5cm단위)': 'height',
    '체중(5kg단위)': 'weight',
    '허리둘레': 'waist',
    '수축기혈압': 'sbp',
    '이완기혈압': 'dbp',
    '식전혈당(공복혈당)': 'fbs',
})
nhis['bmi'] = (nhis['weight'] / ((nhis['height'] / 100) ** 2)).round(1)
nhis_std = nhis[['age', 'gender', 'height', 'weight', 'bmi', 'sbp', 'dbp', 'fbs', 'waist']].copy()
nhis_std['source'] = 'NHIS'
print(f'NHIS:              {len(nhis_std):>7,}명')

In [ ]:
# ── 1-2. Personalized Diet Recommendations ────────────────────
# 타겟 컬럼 보유: Recommended_Calories, Protein, Carbs, Fats, Meal_Plan
diet1 = pd.read_csv(f'{BASE}/data/raw/Personalized_Diet_Recommendations.csv')
diet1_std = pd.DataFrame({
    'age':    diet1['Age'],
    'gender': diet1['Gender'].map({'Male': 1, 'Female': 2}),
    'height': diet1['Height_cm'],
    'weight': diet1['Weight_kg'],
    'bmi':    diet1['BMI'],
    'sbp':    diet1['Blood_Pressure_Systolic'],
    'dbp':    diet1['Blood_Pressure_Diastolic'],
    'fbs':    diet1['Blood_Sugar_Level'],
    'waist':  np.nan,
    'source': 'DIET1',
    # 타겟
    'rec_calories': diet1['Recommended_Calories'],
    'rec_protein':  diet1['Recommended_Protein'],
    'rec_carbs':    diet1['Recommended_Carbs'],
    'rec_fats':     diet1['Recommended_Fats'],
    'meal_plan':    diet1['Recommended_Meal_Plan'],
})
print(f'Personalized Diet: {len(diet1_std):>7,}명')

In [ ]:
# ── 1-3. Diet Recommendations Dataset ────────────────────────
diet2 = pd.read_csv(f'{BASE}/data/raw/diet_recommendations_dataset.csv')
diet2_std = pd.DataFrame({
    'age':    diet2['Age'],
    'gender': diet2['Gender'].map({'Male': 1, 'Female': 2}),
    'height': diet2['Height_cm'],
    'weight': diet2['Weight_kg'],
    'bmi':    diet2['BMI'],
    'sbp':    diet2['Blood_Pressure_mmHg'],
    'dbp':    np.nan,
    'fbs':    diet2['Glucose_mg/dL'],
    'waist':  np.nan,
    'source': 'DIET2',
    'rec_calories': np.nan,
    'rec_protein':  np.nan,
    'rec_carbs':    np.nan,
    'rec_fats':     np.nan,
    'meal_plan':    np.nan,
})
print(f'Diet Recomm:       {len(diet2_std):>7,}명')

In [ ]:
# ── 1-4. Cardiovascular Disease Dataset ──────────────────────
cardio = pd.read_csv(f'{BASE}/data/raw/cardio_train.csv', sep=';')
cardio['bmi'] = (cardio['weight'] / ((cardio['height'] / 100) ** 2)).round(1)
cardio['age_years'] = (cardio['age'] / 365.25).round(0)
cardio_std = pd.DataFrame({
    'age':    cardio['age_years'],
    'gender': cardio['gender'],
    'height': cardio['height'],
    'weight': cardio['weight'],
    'bmi':    cardio['bmi'],
    'sbp':    cardio['ap_hi'],
    'dbp':    cardio['ap_lo'],
    'fbs':    np.nan,
    'waist':  np.nan,
    'source': 'CARDIO',
    'rec_calories': np.nan,
    'rec_protein':  np.nan,
    'rec_carbs':    np.nan,
    'rec_fats':     np.nan,
    'meal_plan':    np.nan,
})
print(f'Cardiovascular:    {len(cardio_std):>7,}명')

In [ ]:
# ── 1-5. Pima Indians Diabetes ───────────────────────────────
diabetes = pd.read_csv(f'{BASE}/data/raw/diabetes.csv')
diabetes_std = pd.DataFrame({
    'age':    diabetes['Age'],
    'gender': 2,  # 전원 여성
    'height': np.nan,
    'weight': np.nan,
    'bmi':    diabetes['BMI'],
    'sbp':    np.nan,
    'dbp':    diabetes['BloodPressure'],
    'fbs':    diabetes['Glucose'],
    'waist':  np.nan,
    'source': 'DIABETES',
    'rec_calories': np.nan,
    'rec_protein':  np.nan,
    'rec_carbs':    np.nan,
    'rec_fats':     np.nan,
    'meal_plan':    np.nan,
})
print(f'Pima Diabetes:     {len(diabetes_std):>7,}명')

In [ ]:
# ── 1-6. 통합 ─────────────────────────────────────────────────
df = pd.concat([nhis_std, diet1_std, diet2_std, cardio_std, diabetes_std], ignore_index=True)
print(f'통합 데이터: {len(df):,}명')
print(df['source'].value_counts())

## 2. 이상치 제거 및 그룹 할당

In [ ]:
OUTLIER_BOUNDS = {
    'sbp':   (60, 300),
    'dbp':   (40, 200),
    'fbs':   (50, 600),
    'bmi':   (10, 60),
    'waist': (40, 200),
}
before = len(df)
for col, (lo, hi) in OUTLIER_BOUNDS.items():
    mask = df[col].isna() | ((df[col] >= lo) & (df[col] <= hi))
    df = df[mask]
print(f'이상치 제거: {before:,} → {len(df):,}명')

In [ ]:
# 정상/주의/위험 변환
def classify_sbp(v):
    if pd.isna(v): return np.nan
    return 0 if v < 120 else (1 if v < 140 else 2)

def classify_dbp(v):
    if pd.isna(v): return np.nan
    return 0 if v < 80 else (1 if v < 90 else 2)

def classify_fbs(v):
    if pd.isna(v): return np.nan
    return 0 if v < 100 else (1 if v < 126 else 2)

def classify_bmi(v):
    if pd.isna(v): return np.nan
    return 0 if v < 23 else (1 if v < 25 else 2)

def classify_waist(row):
    if pd.isna(row['waist']): return np.nan
    return (2 if row['waist'] >= 85 else 0) if row['gender'] == 2 else (2 if row['waist'] >= 90 else 0)

df['sbp_c']   = df['sbp'].apply(classify_sbp)
df['dbp_c']   = df['dbp'].apply(classify_dbp)
df['fbs_c']   = df['fbs'].apply(classify_fbs)
df['bmi_c']   = df['bmi'].apply(classify_bmi)
df['waist_c'] = df.apply(classify_waist, axis=1)

# 규칙 기반 그룹 할당
def assign_group(row):
    hypertension = (row['sbp_c'] >= 1) or (row['dbp_c'] >= 1)
    glucose      = (row['fbs_c'] >= 1)
    obesity      = (row['bmi_c'] >= 1) or (row['waist_c'] >= 1)
    count = sum([hypertension, glucose, obesity])
    if count == 0:                       return '정상군'
    elif count == 3:                     return '복합위험군'
    elif hypertension and glucose:       return '고혈압+혈당이상군'
    elif hypertension and obesity:       return '고혈압+비만군'
    elif glucose and obesity:            return '혈당이상+비만군'
    elif hypertension:                   return '고혈압군'
    elif glucose:                        return '혈당이상군'
    else:                                return '비만군'

df['group'] = df.apply(assign_group, axis=1)
print('그룹 할당 완료')
print(df['group'].value_counts())

## 3. 지도학습 데이터 준비

타겟 컬럼이 있는 Personalized Diet 데이터만 학습에 사용

In [ ]:
# 타겟 있는 데이터만 추출
df_train = df[df['meal_plan'].notna()].copy()
print(f'학습 데이터: {len(df_train):,}명')

FEATURE_COLS = ['age', 'gender', 'height', 'weight', 'bmi',
                'sbp', 'dbp', 'fbs', 'waist',
                'sbp_c', 'dbp_c', 'fbs_c', 'bmi_c', 'waist_c']

TARGET_MEAL   = 'meal_plan'
TARGET_NUTRIENTS = ['rec_calories', 'rec_protein', 'rec_carbs', 'rec_fats']

X = df_train[FEATURE_COLS]

# 식단 플랜 레이블 인코딩
le = LabelEncoder()
y_meal = le.fit_transform(df_train[TARGET_MEAL])

print('\n식단 플랜 목록:')
for i, cls in enumerate(le.classes_):
    print(f'  {i}: {cls}')

## 4. 식단 플랜 분류 모델

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_meal, test_size=0.2, random_state=42, stratify=y_meal
)

# Random Forest: NaN 자체 처리 가능
clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred, average="weighted"):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))

## 5. 영양소 기준 회귀 모델

In [ ]:
regressors = {}

for target in TARGET_NUTRIENTS:
    y_reg = df_train[target]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y_reg, test_size=0.2, random_state=42
    )
    reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    reg.fit(X_tr, y_tr)
    y_pr = reg.predict(X_te)
    regressors[target] = reg
    print(f'{target:15s}: MAE={mean_absolute_error(y_te, y_pr):.2f}  R2={r2_score(y_te, y_pr):.4f}')

## 6. 저장

In [ ]:
MODEL_DIR = f'{BASE}/models/diet'
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(clf,          f'{MODEL_DIR}/meal_plan_classifier.pkl')
joblib.dump(le,           f'{MODEL_DIR}/label_encoder.pkl')
for target, reg in regressors.items():
    joblib.dump(reg, f'{MODEL_DIR}/{target}_regressor.pkl')

print('저장 완료')

## 7. 예측 함수 테스트

In [ ]:
def predict(age, gender, height, weight, sbp, dbp, fbs, waist=np.nan):
    bmi     = round(weight / ((height / 100) ** 2), 1)
    sbp_c   = 0 if sbp < 120 else (1 if sbp < 140 else 2)
    dbp_c   = 0 if dbp < 80  else (1 if dbp < 90  else 2)
    fbs_c   = 0 if fbs < 100 else (1 if fbs < 126 else 2)
    bmi_c   = 0 if bmi < 23  else (1 if bmi < 25  else 2)
    waist_c = np.nan if pd.isna(waist) else (2 if (waist >= 85 if gender == 2 else waist >= 90) else 0)

    input_data = pd.DataFrame([{
        'age': age, 'gender': gender, 'height': height, 'weight': weight, 'bmi': bmi,
        'sbp': sbp, 'dbp': dbp, 'fbs': fbs, 'waist': waist,
        'sbp_c': sbp_c, 'dbp_c': dbp_c, 'fbs_c': fbs_c, 'bmi_c': bmi_c, 'waist_c': waist_c
    }])

    meal_plan = le.inverse_transform(clf.predict(input_data))[0]
    calories  = regressors['rec_calories'].predict(input_data)[0]
    protein   = regressors['rec_protein'].predict(input_data)[0]
    carbs     = regressors['rec_carbs'].predict(input_data)[0]
    fats      = regressors['rec_fats'].predict(input_data)[0]

    print(f'식단 플랜    : {meal_plan}')
    print(f'권장 칼로리  : {calories:.0f} kcal')
    print(f'권장 단백질  : {protein:.1f} g')
    print(f'권장 탄수화물: {carbs:.1f} g')
    print(f'권장 지방    : {fats:.1f} g')

print('=== 테스트 1: 45세 남성 정상 수치 ===')
predict(age=45, gender=1, height=175, weight=70, sbp=115, dbp=75, fbs=90)

print('\n=== 테스트 2: 55세 여성 고혈압+비만 ===')
predict(age=55, gender=2, height=160, weight=75, sbp=145, dbp=92, fbs=95, waist=88)

## 8. 결과 요약

In [ ]:
print('=' * 60)
print('분류 모델 결과 요약')
print('=' * 60)
print(f'모델           : Random Forest')
print(f'Accuracy       : {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 Score       : {f1_score(y_test, y_pred, average="weighted"):.4f}')
print(f'입력 피처      : {FEATURE_COLS}')
print(f'출력 (분류)    : 식단 플랜 ({list(le.classes_)})')
print(f'출력 (회귀)    : 권장 칼로리, 단백질, 탄수화물, 지방')
print()
print('→ 다음 단계: RAG 문서 검색 + LLM 프롬프트 구성')
print('=' * 60)